# FactCG as a native MiniCheck model — Kaggle smoke test

This notebook pins MiniCheck and the official FactCG checkpoint, applies a guarded patch to the existing Inferencer branch, and exercises only the public MiniCheck.score() API. It keeps MiniCheck's chunking, max-over-chunks aggregation, output contract, and strict probability > 0.5 threshold.

Kaggle: enable a T4 GPU and Internet. For the gated LLM-AggreFact dataset, accept its Hugging Face access conditions and add HF_TOKEN as a Kaggle Secret.

In [ ]:
# Reproducible Kaggle setup: install, clone, and pin before importing ML packages.
import os
import subprocess
import sys
from pathlib import Path

MINICHECK_REPO = "https://github.com/Liyan06/MiniCheck.git"
MINICHECK_COMMIT = "b58b9fa69acbd1015ec970fa65dd752413a053d2"
FACTCG_MODEL_ID = "yaxili96/FactCG-DeBERTa-v3-Large"
FACTCG_MODEL_REVISION = "0430e3509dbd28d2dff7a117c0eae25359ff3e80"
FACTCG_SOURCE_COMMIT = "41f185413312ed9b3904ec03963fe899693498e0"
DATASET_ID = "lytang/LLM-AggreFact"
DATASET_REVISION = "981dfd0"
WORKING_DIR = Path("/kaggle/working")
MINICHECK_DIR = WORKING_DIR / "MiniCheck"
CACHE_DIR = WORKING_DIR / "ckpts"

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

packages = [
    "transformers==4.38.1",
    "accelerate==0.27.2",
    "datasets==2.18.0",
    "huggingface_hub==0.21.4",
    "sentencepiece==0.2.0",
    "nltk==3.8.1",
    "scikit-learn==1.4.2",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *packages], check=True)

if not (MINICHECK_DIR / ".git").exists():
    subprocess.run(["git", "clone", MINICHECK_REPO, str(MINICHECK_DIR)], check=True)
subprocess.run(["git", "-C", str(MINICHECK_DIR), "fetch", "origin", MINICHECK_COMMIT], check=True)
subprocess.run(
    ["git", "-C", str(MINICHECK_DIR), "checkout", "--force", "--detach", MINICHECK_COMMIT],
    check=True,
)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(MINICHECK_DIR)], check=True)
print("Pinned MiniCheck:", subprocess.check_output(
    ["git", "-C", str(MINICHECK_DIR), "rev-parse", "HEAD"], text=True
).strip())

# Optional gated-dataset authentication. Add HF_TOKEN under Kaggle Add-ons > Secrets.
def get_hf_token():
    token = os.environ.get("HF_TOKEN")
    if token:
        return token
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return None

HF_TOKEN = get_hf_token()
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("HF_TOKEN loaded from the environment/Kaggle Secrets.")
else:
    print("HF_TOKEN not found. The public model can load anonymously; the gated dataset requires an accepted-access token.")


In [ ]:
# Apply a guarded, minimal patch to MiniCheck's existing Inferencer branch.
from pathlib import Path

inference_path = MINICHECK_DIR / "minicheck" / "inference.py"
minicheck_path = MINICHECK_DIR / "minicheck" / "minicheck.py"

def replace_once(text, old, new, description):
    count = text.count(old)
    if count != 1:
        raise RuntimeError(f"{description}: expected exactly one source match, found {count}")
    return text.replace(old, new, 1)

text = inference_path.read_text(encoding="utf-8")
text = replace_once(
    text,
    "from typing import List\n",
    """from typing import List

# FactCG contract verified against derenlei/FactCG@41f1854:
# - official HF checkpoint/tokenizer
# - instruction template from factcg/utils.py
# - class 1 is the supported/entailed probability in factcg/inference.py
FACTCG_MODEL_ID = "yaxili96/FactCG-DeBERTa-v3-Large"
FACTCG_MODEL_REVISION = "0430e3509dbd28d2dff7a117c0eae25359ff3e80"
FACTCG_INSTRUCTION_TEMPLATE = (
    '{text_a}\\n\\nChoose your answer: based on the paragraph above can we conclude that '
    '"{text_b}"?\\n\\nOPTIONS:\\n- Yes\\n- No\\nI think the answer is '
)
""",
    "add FactCG constants",
)
text = replace_once(
    text,
    """            elif model_name == 'deberta-v3-large':
                ckpt = 'lytang/MiniCheck-DeBERTa-v3-Large'
                self.max_model_len=2048 if max_model_len is None else max_model_len
                
            else:
""",
    """            elif model_name == 'deberta-v3-large':
                ckpt = 'lytang/MiniCheck-DeBERTa-v3-Large'
                self.max_model_len=2048 if max_model_len is None else max_model_len

            elif model_name == 'FactCG-DeBERTa-v3-Large':
                ckpt = FACTCG_MODEL_ID
                self.max_model_len=2048 if max_model_len is None else max_model_len
                
            else:
""",
    "add FactCG checkpoint branch",
)
text = replace_once(
    text,
    """            config = AutoConfig.from_pretrained(ckpt, num_labels=2, finetuning_task="text-classification", revision='main', token=None, cache_dir=cache_dir)
""",
    """            revision = FACTCG_MODEL_REVISION if model_name == 'FactCG-DeBERTa-v3-Large' else 'main'
            config = AutoConfig.from_pretrained(ckpt, num_labels=2, finetuning_task="text-classification", revision=revision, token=None, cache_dir=cache_dir)
""",
    "pin FactCG config revision",
)
if text.count("revision='main'") != 2:
    raise RuntimeError("Expected two remaining tokenizer/model revision='main' occurrences")
text = text.replace("revision='main'", "revision=revision")
text = replace_once(
    text,
    """        self.model.eval()
        self.batch_size = batch_size
""",
    """        if model_name == 'FactCG-DeBERTa-v3-Large':
            if config.num_labels != 2:
                raise ValueError(f"FactCG must expose two labels, got {config.num_labels}")
            config_id2label = {int(k): str(v) for k, v in config.id2label.items()}
            semantic_support_ids = [
                idx for idx, label in config_id2label.items()
                if label.lower() in {'supported', 'support', 'entailment', 'entailed', 'yes'}
            ]
            if semantic_support_ids and semantic_support_ids != [1]:
                raise ValueError(f"FactCG config conflicts with official class-1 support mapping: {config_id2label}")
            self.support_label_id = 1
            self.support_label_source = (
                "derenlei/FactCG@41f1854 factcg/inference.py softmax(logits)[:, 1]; "
                "training orig_label=1 denotes supported"
            )
        else:
            # Preserve the established MiniCheck encoder behavior.
            self.support_label_id = 1
            self.support_label_source = "MiniCheck legacy encoder contract"

        self.model.eval()
        self.batch_size = batch_size
""",
    "add source-backed support-label mapping",
)
text = replace_once(
    text,
    """        support_prob_per_chunk = label_probs[:, 1].cpu().numpy()
        max_support_prob = label_probs[:, 1].max().item()
""",
    """        support_prob_per_chunk = label_probs[:, self.support_label_id].cpu().numpy()
        max_support_prob = label_probs[:, self.support_label_id].max().item()
""",
    "route probabilities through verified support class",
)
text = replace_once(
    text,
    """        original_text = [self.tokenizer.eos_token.join([one_doc, one_claim]) for one_doc, one_claim in zip(doc, claim)]

        batch_input = []
""",
    """        if self.model_name == 'FactCG-DeBERTa-v3-Large':
            original_text = [
                FACTCG_INSTRUCTION_TEMPLATE.format(text_a=one_doc, text_b=one_claim)
                for one_doc, one_claim in zip(doc, claim)
            ]
            # This matches the official FactCG tokenizer setup.
            self.tokenizer.pad_token = self.tokenizer.eos_token
        else:
            original_text = [self.tokenizer.eos_token.join([one_doc, one_claim]) for one_doc, one_claim in zip(doc, claim)]

        batch_input = []
""",
    "use official FactCG input formatting",
)
text = replace_once(
    text,
    """            else:
                model_inputs = self.tokenizer(
                    [text for text in mini_batch], 
                    max_length=self.max_model_len, 
                    truncation=True, 
                    padding=True, 
                    return_tensors="pt"
                ) 
""",
    """            elif self.model_name == 'FactCG-DeBERTa-v3-Large':
                model_inputs = self.tokenizer(
                    [text for text in mini_batch],
                    max_length=self.max_model_len,
                    truncation='only_first',
                    padding='longest',
                    return_tensors="pt"
                )
            else:
                model_inputs = self.tokenizer(
                    [text for text in mini_batch], 
                    max_length=self.max_model_len, 
                    truncation=True, 
                    padding=True, 
                    return_tensors="pt"
                ) 
""",
    "use official FactCG tokenizer arguments",
)
text = replace_once(
    text,
    """            batch_org_chunks.append([item[:item.find('</s>')] for item in mini_batch]) 
""",
    """            if self.model_name == 'FactCG-DeBERTa-v3-Large':
                # The prompt has no separator from which to recover the document.
                start = len(batch_org_chunks) * self.batch_size
                batch_org_chunks.append(list(doc[start:start + len(mini_batch)]))
            else:
                batch_org_chunks.append([item[:item.find('</s>')] for item in mini_batch]) 
""",
    "preserve native used_chunks output for FactCG",
)
inference_path.write_text(text, encoding="utf-8")

text = minicheck_path.read_text(encoding="utf-8")
text = replace_once(
    text,
    "            - 'flan-t5-large'\n            - 'Bespoke-MiniCheck-7B'",
    "            - 'flan-t5-large'\n            - 'FactCG-DeBERTa-v3-Large'\n            - 'Bespoke-MiniCheck-7B'",
    "document FactCG model option",
)
text = replace_once(
    text,
    "                - 'flan-t5-large'\n                    Default: 2048\n                - 'Bespoke-MiniCheck-7B'",
    "                - 'flan-t5-large'\n                    Default: 2048\n                - 'FactCG-DeBERTa-v3-Large'\n                    Default: 2048\n                - 'Bespoke-MiniCheck-7B'",
    "document FactCG maximum length",
)
text = replace_once(
    text,
    "            - 'flan-t5-large'\n                Default: 500\n            - 'Bespoke-MiniCheck-7B'",
    "            - 'flan-t5-large'\n                Default: 500\n            - 'FactCG-DeBERTa-v3-Large'\n                Default: 400\n            - 'Bespoke-MiniCheck-7B'",
    "document FactCG chunk default",
)
old_all = "['roberta-large', 'deberta-v3-large', 'flan-t5-large', 'Bespoke-MiniCheck-7B', 'Granite-Guardian-3.3-8B']"
new_all = "['roberta-large', 'deberta-v3-large', 'flan-t5-large', 'FactCG-DeBERTa-v3-Large', 'Bespoke-MiniCheck-7B', 'Granite-Guardian-3.3-8B']"
if text.count(old_all) != 2:
    raise RuntimeError(f"Expected two complete supported-model lists, found {text.count(old_all)}")
text = text.replace(old_all, new_all)
text = replace_once(
    text,
    "if model_name in ['roberta-large', 'deberta-v3-large', 'flan-t5-large']:",
    "if model_name in ['roberta-large', 'deberta-v3-large', 'flan-t5-large', 'FactCG-DeBERTa-v3-Large']:",
    "route FactCG through Inferencer",
)
minicheck_path.write_text(text, encoding="utf-8")

# Syntax-check the patched package before any model download.
import py_compile
py_compile.compile(str(inference_path), doraise=True)
py_compile.compile(str(minicheck_path), doraise=True)
print("MiniCheck patched successfully:", inference_path)


## Load through MiniCheck and validate handcrafted cases

FactCG's official source uses its instruction prompt and class index 1 as the supported probability. The checkpoint config's labels are generic, so the patch validates the binary config and records the official-source provenance instead of inferring semantics from LABEL_1.

In [ ]:
import gc
import json
import platform
from importlib.metadata import version
from pathlib import Path

import nltk
import numpy as np
import pandas as pd
import torch

nltk.download("punkt", quiet=True)

from minicheck.minicheck import MiniCheck

MODEL_NAME = "FactCG-DeBERTa-v3-Large"
BATCH_SIZE = 8
CHUNK_SIZE = 500
OUTPUT_DIR = Path("/kaggle/working/factcg_smoke")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

scorer = MiniCheck(
    model_name=MODEL_NAME,
    batch_size=BATCH_SIZE,
    cache_dir=str(CACHE_DIR),
)
assert scorer.model.model_name == MODEL_NAME
assert scorer.model.support_label_id == 1
print("Checkpoint config id2label:", scorer.model.model.config.id2label)
print("Support-label provenance:", scorer.model.support_label_source)
print("Tokenizer:", type(scorer.model.tokenizer).__name__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

filler = "This sentence contains neutral background material. " * 90
handcrafted = pd.DataFrame([
    {
        "case": "paris_supported",
        "doc": "Paris is the capital of France.",
        "claim": "Paris is France's capital.",
        "expected_label": 1,
    },
    {
        "case": "paris_unsupported",
        "doc": "Paris is the capital of France.",
        "claim": "Berlin is the capital of France.",
        "expected_label": 0,
    },
    {
        "case": "boiling_supported",
        "doc": "At standard atmospheric pressure, pure water boils at 100 degrees Celsius.",
        "claim": "Pure water boils at 100 degrees Celsius at standard pressure.",
        "expected_label": 1,
    },
    {
        "case": "boiling_unsupported",
        "doc": "At standard atmospheric pressure, pure water boils at 100 degrees Celsius.",
        "claim": "Pure water boils at 40 degrees Celsius at standard pressure.",
        "expected_label": 0,
    },
    {
        "case": "multi_chunk_evidence",
        "doc": filler + " The observatory confirmed that Kepler-186f orbits a red dwarf star.",
        "claim": "Kepler-186f orbits a red dwarf star.",
        "expected_label": 1,
    },
])

preds, probs, used_chunks, chunk_probs = scorer.score(
    docs=handcrafted["doc"].tolist(),
    claims=handcrafted["claim"].tolist(),
    chunk_size=CHUNK_SIZE,
)

n = len(handcrafted)
assert len(preds) == len(probs) == len(used_chunks) == len(chunk_probs) == n
assert all(isinstance(x, (int, np.integer)) and int(x) in (0, 1) for x in preds)
assert all(np.isfinite(float(p)) and 0.0 <= float(p) <= 1.0 for p in probs)
assert all(int(pred) == int(float(prob) > 0.5) for pred, prob in zip(preds, probs))
assert all(len(chunks) == len(np.asarray(per_chunk).reshape(-1)) for chunks, per_chunk in zip(used_chunks, chunk_probs))
assert len(used_chunks[-1]) > 1, "The long handcrafted document should exercise MiniCheck chunking."

handcrafted["prediction"] = preds
handcrafted["support_probability"] = probs
handcrafted["num_chunks"] = [len(x) for x in used_chunks]
handcrafted["chunk_probabilities"] = [
    json.dumps(np.asarray(x).reshape(-1).astype(float).tolist()) for x in chunk_probs
]
handcrafted["matches_expected"] = handcrafted["prediction"] == handcrafted["expected_label"]
handcrafted.to_csv(OUTPUT_DIR / "handcrafted_results.csv", index=False)
display(handcrafted[["case", "expected_label", "prediction", "support_probability", "num_chunks", "matches_expected"]])


## One deterministic test example from each LLM-AggreFact dataset

This is a pipeline smoke test, not a benchmark estimate. It preserves each row's original test-split index and uses the same chunk_size=500, batch_size=8, strict fixed threshold, and MiniCheck output contract as the sample experiment.

In [ ]:
from datasets import load_dataset

if not HF_TOKEN:
    raise RuntimeError(
        "LLM-AggreFact is gated. Accept access on Hugging Face, add HF_TOKEN to Kaggle Secrets, and Run All."
    )

raw_test = load_dataset(
    DATASET_ID,
    split="test",
    revision=DATASET_REVISION,
    token=HF_TOKEN,
)
full_df = raw_test.to_pandas().reset_index(names="source_index")
dataset_names = sorted(full_df["dataset"].unique())
assert len(dataset_names) == 11, f"Expected 11 datasets, found {len(dataset_names)}: {dataset_names}"

tiny_df = pd.concat(
    [
        full_df.loc[full_df["dataset"].eq(name)].sample(n=1, random_state=42)
        for name in dataset_names
    ],
    ignore_index=True,
).sort_values(["dataset", "source_index"]).reset_index(drop=True)

tiny_preds, tiny_probs, tiny_chunks, tiny_chunk_probs = scorer.score(
    docs=tiny_df["doc"].tolist(),
    claims=tiny_df["claim"].tolist(),
    chunk_size=CHUNK_SIZE,
)

assert len(tiny_df) == len(tiny_preds) == len(tiny_probs) == len(tiny_chunks) == len(tiny_chunk_probs) == 11
assert tiny_df["dataset"].nunique() == 11
assert tiny_df["source_index"].notna().all()
assert all(np.isfinite(float(p)) and 0.0 <= float(p) <= 1.0 for p in tiny_probs)
assert all(int(pred) == int(float(prob) > 0.5) for pred, prob in zip(tiny_preds, tiny_probs))
assert all(len(chunks) >= 1 for chunks in tiny_chunks)
assert all(len(chunks) == len(np.asarray(cp).reshape(-1)) for chunks, cp in zip(tiny_chunks, tiny_chunk_probs))

tiny_df["prediction"] = tiny_preds
tiny_df["support_probability"] = tiny_probs
tiny_df["num_chunks"] = [len(x) for x in tiny_chunks]
tiny_df["chunk_probabilities"] = [
    json.dumps(np.asarray(x).reshape(-1).astype(float).tolist()) for x in tiny_chunk_probs
]
tiny_df["correct"] = tiny_df["prediction"].astype(int) == tiny_df["label"].astype(int)
tiny_df.to_csv(OUTPUT_DIR / "llm_aggrefact_11_dataset_smoke.csv", index=False)
display(tiny_df[["dataset", "source_index", "label", "prediction", "support_probability", "num_chunks", "correct"]])


In [ ]:
import hashlib
import subprocess
from datetime import datetime, timezone

def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

diagnostics = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "model_name": MODEL_NAME,
    "model_id": FACTCG_MODEL_ID,
    "model_revision": FACTCG_MODEL_REVISION,
    "factcg_source_commit": FACTCG_SOURCE_COMMIT,
    "minicheck_commit": MINICHECK_COMMIT,
    "dataset_id": DATASET_ID,
    "dataset_revision": DATASET_REVISION,
    "split": "test",
    "batch_size": BATCH_SIZE,
    "chunk_size": CHUNK_SIZE,
    "threshold_rule": "prediction = 1 iff support_probability > 0.5",
    "support_label_id": scorer.model.support_label_id,
    "support_label_source": scorer.model.support_label_source,
    "checkpoint_id2label": {str(k): str(v) for k, v in scorer.model.model.config.id2label.items()},
    "handcrafted_contract_checks_passed": True,
    "all_11_datasets_covered": tiny_df["dataset"].nunique() == 11,
    "python": platform.python_version(),
    "torch": torch.__version__,
    "transformers": version("transformers"),
    "datasets": version("datasets"),
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "patched_inference_sha256": sha256_file(inference_path),
    "patched_minicheck_sha256": sha256_file(minicheck_path),
}
(OUTPUT_DIR / "diagnostics.json").write_text(json.dumps(diagnostics, indent=2), encoding="utf-8")
print("Smoke test complete. Saved:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(" -", path)
